# Genetic correlation between diseases

A disease-by-disease genetic correlation matrix, used by Supplementary Results 14 to ask how
much of the pleiotropy counts is carried by correlated diseases. Built from the pairwise LDSC
table (an external run; only its output is available, see GAPS.md): study pairs are mapped to
disease pairs through the study index, pairs with fewer than 100,000 SNPs are dropped, and the
pair with the smallest standard error is kept where a disease pair is measured more than once.

Writes `rg_matrix`.

In [ ]:
import numpy as np
import pandas as pd
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from pyspark.sql import Window
from pyspark.sql import functions as f

from manuscript_methods import paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

MIN_SNPS = 100_000

In [ ]:
rg = session.spark.read.parquet(paper.baseline("canonical_pairwise_table") + "/canonical_pairwise_table.parquet")
studies = StudyIndex.from_parquet(session, paper.release("study")).df.select("studyId", "diseaseIds")
print("study pairs:", rg.count())

pairs = (
    rg.join(studies.alias("a"), f.col("studyId1") == f.col("a.studyId"), "left")
    .join(studies.alias("b"), f.col("studyId2") == f.col("b.studyId"), "left")
    .select(*rg.columns, f.col("a.diseaseIds").alias("diseaseIds1"), f.col("b.diseaseIds").alias("diseaseIds2"))
    .withColumn("diseaseId1", f.explode("diseaseIds1"))
    .withColumn("diseaseId2", f.explode("diseaseIds2"))
    .filter(f.col("rg").isNotNull() & f.col("rg_se").isNotNull())
    .filter(f.col("n_snps_used") >= MIN_SNPS)
    .withColumn("lo", f.least("diseaseId1", "diseaseId2"))
    .withColumn("hi", f.greatest("diseaseId1", "diseaseId2"))
)

deduped = (
    pairs.withColumn("rank", f.row_number().over(Window.partitionBy("lo", "hi").orderBy(f.asc("rg_se"))))
    .filter(f.col("rank") == 1)
    .select("lo", "hi", "rg")
)
print("disease pairs:", deduped.count())

In [ ]:
observed = deduped.toPandas()
diseases = sorted(set(observed["lo"]) | set(observed["hi"]))
index = {disease: i for i, disease in enumerate(diseases)}

matrix = np.zeros((len(diseases), len(diseases)))
rows = observed["lo"].map(index).to_numpy()
cols = observed["hi"].map(index).to_numpy()
values = np.nan_to_num(np.clip(observed["rg"].to_numpy(), -1.0, 1.0), nan=0.0)
matrix[rows, cols] = values
matrix[cols, rows] = values
np.fill_diagonal(matrix, 1.0)

labelled = pd.DataFrame(matrix, index=diseases, columns=diseases)
labelled.to_parquet(paper.derived("rg_matrix"))
print("traits in the matrix:", len(diseases))
print("symmetric:", np.allclose(matrix, matrix.T), "| range:", matrix.min(), matrix.max())